<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeshengliu213/ai-for-materials/blob/main/09%20-%20Generative%20Models/solutions/09_exercises_solutions.ipynb)

# 09d · 基础习题集参考解答：SELFIES、变分自编码器、代理模型防坑与扩散模型

本 Notebook 提供 `09d_exercises.ipynb` 中全部 8 道基础编程练习的完整参考实现与输出验证。

---
## 习题 1 解答：SELFIES 编码、符号切分与单点突变合法性验证

In [ ]:
import selfies as sf
from rdkit import Chem

def selfies_roundtrip_and_mutate(smiles, replacement_token="[=O]"):
    encoded = sf.encoder(smiles)
    tokens = list(sf.split_selfies(encoded))
    n_tok = len(tokens)
    tokens[n_tok // 2] = replacement_token
    mutated_smiles = sf.decoder("".join(tokens))
    is_valid = (Chem.MolFromSmiles(mutated_smiles) is not None) if mutated_smiles else False
    return n_tok, mutated_smiles, is_valid

n_t, mut_s, ok = selfies_roundtrip_and_mutate("c1ccc2c(c1)sc1c3ccccc3sc21")
print(f"Token count: {n_t} | Mutated SMILES: {mut_s} | RDKit Valid: {ok}")

---
## 习题 2 解答：手写 VAE 重参数化技巧与对角高斯 KL 散度

In [ ]:
import torch

def reparameterize(mu, logvar, eps):
    std = torch.exp(0.5 * logvar)
    return mu + std * eps

def kl_divergence(mu, logvar):
    kl_per_sample = -0.5 * torch.sum(1.0 + logvar - mu.pow(2) - logvar.exp(), dim=-1)
    return float(kl_per_sample.mean().item())

z_mu = torch.zeros(4, 16)
z_lv = torch.zeros(4, 16)
eps = torch.ones(4, 16) * 0.5
print("Reparameterized mean:", reparameterize(z_mu, z_lv, eps).mean().item())
print("KL(N(0,I) || N(0,I)):", kl_divergence(z_mu, z_lv))
print("KL(N(1,I) || N(0,I)):", kl_divergence(torch.ones(4, 16), z_lv))

---
## 习题 3 解答：高维潜空间中的直线插值（Lerp）范数塌陷 vs. 球面线性插值（Slerp）

In [ ]:
import torch

def slerp_midpoint(z_a, z_b):
    a_n = z_a / z_a.norm()
    b_n = z_b / z_b.norm()
    omega = torch.acos((a_n * b_n).sum().clamp(-1.0 + 1e-6, 1.0 - 1e-6))
    r = 0.5 * (z_a.norm() + z_b.norm())
    return r * (torch.sin(0.5 * omega) * a_n + torch.sin(0.5 * omega) * b_n) / torch.sin(omega)

def compare_midpoint_norms(z_a, z_b):
    lerp_mid = 0.5 * (z_a + z_b)
    slerp_mid = slerp_midpoint(z_a, z_b)
    return float(lerp_mid.norm().item()), float(slerp_mid.norm().item())

torch.manual_seed(0)
a = torch.randn(128); a = a / a.norm() * 10.0
b = torch.randn(128); b = b / b.norm() * 10.0
n_lerp, n_slerp = compare_midpoint_norms(a, b)
print(f"Endpoint norm: 10.00 | Lerp midpoint norm: {n_lerp:.2f} (collapsed!) | Slerp midpoint norm: {n_slerp:.2f}")

---
## 习题 4 解答：高维高斯球壳半径与 VAE 先验空洞（Prior Hole）定量计算

In [ ]:
import torch

def gaussian_shell_radii(d=128, posterior_std=0.55, n_samples=5000):
    torch.manual_seed(0)
    z_prior = torch.randn(n_samples, d)
    z_post = z_prior * posterior_std
    r_prior = float(z_prior.norm(dim=1).median().item())
    r_post = float(z_post.norm(dim=1).median().item())
    return round(r_prior, 2), round(r_post, 2), round(r_post / r_prior, 3)

r_p, r_q, ratio = gaussian_shell_radii(128, 0.55)
print(f"Prior shell median radius: {r_p} | Posterior shell median radius: {r_q} | Ratio: {ratio}")

---
## 习题 5 解答：二值 Morgan 指纹的比特饱和（Bit Saturation）实验

In [ ]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

def count_oligomer_on_bits(monomer_smiles="c1ccc2nsnc2c1", lengths=(1, 2, 3, 6, 10)):
    fp_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
    result = {}
    for k in lengths:
        smi = "-".join([monomer_smiles] * k)
        mol = Chem.MolFromSmiles(smi)
        arr = fp_gen.GetFingerprintAsNumPy(mol)
        result[k] = int(arr.sum())
    return result

print("Active bits in 2048-bit binary Morgan fingerprint vs. chain length k:")
for k, cnt in count_oligomer_on_bits().items():
    print(f"  k = {k:2d} units -> {cnt} active bits")

---
## 习题 6 解答：结合适用域过滤与悲观置信下界（LCB）的候选材料筛选器

In [ ]:
import numpy as np

def select_best_candidate_lcb(tree_preds, max_sims, sim_threshold=0.40, kappa=2.0):
    means = tree_preds.mean(axis=1)
    stds = tree_preds.std(axis=1)
    lcb = means - kappa * stds
    valid_mask = max_sims >= sim_threshold
    masked_lcb = np.where(valid_mask, lcb, -np.inf)
    best_idx = int(np.argmax(masked_lcb))
    return best_idx, float(lcb[best_idx])

fake_preds = np.array([
    [8.0, 4.0, 9.0, 3.0],   # 均值 6.0，但树间分歧大 (std=2.55 -> LCB=0.90)
    [6.5, 6.4, 6.6, 6.5],   # 均值 6.5，树间高度一致 (std=0.07 -> LCB=6.36)，且在适用域内
    [7.2, 7.0, 7.1, 7.3],   # 均值 7.15，但 sim=0.25 < 0.40 超出适用域被拒
])
fake_sims = np.array([0.55, 0.62, 0.25])
idx, score = select_best_candidate_lcb(fake_preds, fake_sims)
print(f"Selected Candidate Index: #{idx} | Pessimistic LCB Score: {score:.2f}%")

---
## 习题 7 解答：电镜超分辨率图像的峰值信噪比（PSNR）计算

In [ ]:
import math
import torch

def compute_psnr_db(pred, target):
    mse = torch.mean((pred - target) ** 2).item()
    return float(-10.0 * math.log10(mse + 1e-12))

gt = torch.ones(1, 1, 16, 16) * 0.5
p1 = gt + 0.10
p2 = gt + 0.05
print(f"PSNR (error = 0.10): {compute_psnr_db(p1, gt):.2f} dB")
print(f"PSNR (error = 0.05): {compute_psnr_db(p2, gt):.2f} dB (+6.02 dB gain when error is halved)")

---
## 习题 8 解答：去噪扩散模型（DDPM）的前向闭合加噪与信噪比计算

In [ ]:
import torch

def ddpm_forward_step(x0, t_index, betas, noise):
    alphas_bar = torch.cumprod(1.0 - betas, dim=0)
    ab_t = alphas_bar[t_index]
    x_t = torch.sqrt(ab_t) * x0 + torch.sqrt(1.0 - ab_t) * noise
    snr = ab_t / (1.0 - ab_t)
    return x_t, float(snr.item())

betas_demo = torch.linspace(1e-4, 0.08, 40)
x0_demo = torch.ones(1, 1, 8, 8)
eps_demo = torch.zeros(1, 1, 8, 8)
xt_out, snr_val = ddpm_forward_step(x0_demo, 10, betas_demo, eps_demo)
print(f"Signal scale sqrt(alpha_bar_10) = {xt_out.mean().item():.4f} | SNR(t=10) = {snr_val:.4f}")